# Step 5 — BAGEL exact CFG branch-pruning performance

This notebook measures the optimization that passed Step 4's numerical gate: `drop_image` is bit-identical to `reference` under the default text-to-image settings. It runs five paired repeats on one loaded server, alternates policy order, warms both execution shapes, disables latent capture, and requires exact final PNG equality in every pair.

The scope is intentionally **eager B=1**. Compilation remains disabled because the dynamic compiled path has not passed correctness. This experiment measures branch-pruning performance, not continuous batching.

## 1. Mount Drive and define the run

Use a Colab **A100 High-RAM** runtime. The large Hugging Face checkpoint cache persists on Drive. The smaller `uv` cache stays on `/content` because Google Drive's FUSE filesystem does not reliably support `uv`'s cache operations.

In [ ]:
from google.colab import drive
from datetime import datetime, timezone
from pathlib import Path
import json, os, shutil, socket, subprocess, sys

drive.mount('/content/drive')
RUN_ID = 'cfg_branch_perf_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
FORK_URL = 'https://github.com/yegurla-sumanth/mstar.git'
RUN_COMMIT = 'b22e8bc2be7297e6cf636d369f442923c73a5d70'
REPO = Path('/content') / f'mstar_{RUN_ID}'
DRIVE_ROOT = Path('/content/drive/MyDrive/CMU_11868_BAGEL')
CACHE_DIR = DRIVE_ROOT / 'persistent_cache/hf_cache_bagel'
UV_CACHE_DIR = Path('/content/uv_cache')
RESULT_DIR = DRIVE_ROOT / RUN_ID
BENCHMARK_DIR = RESULT_DIR / 'cfg_branch_performance_b1_eager'
RESULT_DIR.mkdir(parents=True, exist_ok=False)
CACHE_DIR.mkdir(parents=True, exist_ok=True)
UV_CACHE_DIR.mkdir(parents=True, exist_ok=True)
drive_free_gib = shutil.disk_usage(DRIVE_ROOT).free / (1024 ** 3)
assert drive_free_gib >= 45, f'Need at least 45 GiB free on Drive; found {drive_free_gib:.1f}'
os.environ['UV_CACHE_DIR'] = str(UV_CACHE_DIR)
print({'run_id': RUN_ID, 'repo': str(REPO), 'results': str(RESULT_DIR)})

## 2. Clone the pinned implementation and verify the A100

The hardware preflight stops before environment installation or checkpoint access if this is not an 80 GB-class runtime.

In [ ]:
subprocess.run(['git', 'clone', '--no-checkout', FORK_URL, str(REPO)], check=True)
subprocess.run(['git', 'checkout', '--detach', RUN_COMMIT], cwd=REPO, check=True)
head = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip()
assert head == RUN_COMMIT, (head, RUN_COMMIT)
source = {
    'fork_url': FORK_URL,
    'requested_commit': RUN_COMMIT,
    'resolved_commit': head,
    'git_status': subprocess.check_output(['git', 'status', '--porcelain=v1'], cwd=REPO, text=True),
}
(RESULT_DIR / 'source.json').write_text(json.dumps(source, indent=2) + '\n')
subprocess.run([
    sys.executable, str(REPO / 'experiments/bagel/scripts/environment_probe.py'),
    '--output', str(RESULT_DIR / 'preinstall_environment.json'),
    '--disk-path', '/content',
    '--require-gpu-memory-gib', '75',
    '--require-host-memory-gib', '50',
    '--require-disk-free-gib', '55',
], check=True)
print('Hardware preflight passed at commit', head)

## 3. Install and validate the pinned environment

A fresh Colab VM still needs its virtual environment. The package cache is deliberately local; only the much larger model checkpoint is persisted on Drive.

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'uv==0.10.0'], check=True)
UV = subprocess.check_output([sys.executable, '-c', 'import shutil; print(shutil.which("uv"))'], text=True).strip()
VENV = REPO / '.venv-bagel'
subprocess.run([UV, 'venv', '--python', '3.12', '--seed', str(VENV)], check=True)
VENV_PY = VENV / 'bin/python'
constraints = REPO / 'experiments/bagel/constraints-colab-cu128.txt'
subprocess.run([
    UV, 'pip', 'install', '--python', str(VENV_PY), '--torch-backend=cu128',
    '--constraint', str(constraints), '-e', f'{REPO}[bagel]', 'matplotlib==3.10.7',
], check=True)
subprocess.run([UV, 'pip', 'check', '--python', str(VENV_PY)], check=True)
ninja_path = VENV / 'bin/ninja'
assert ninja_path.exists(), f'Missing required FlashInfer JIT tool: {ninja_path}'
subprocess.run([str(ninja_path), '--version'], check=True)
for test_path in (
    'test/modular/test_bagel_checkpoint_revision.py',
    'test/modular/test_execution_trace.py',
    'test/modular/test_bagel_image_batch_packing.py',
):
    subprocess.run([str(VENV_PY), str(REPO / test_path), '-v'], cwd=REPO, check=True)
import_check = subprocess.check_output([
    str(VENV_PY), '-c',
    'import os; os.environ["MPLBACKEND"]="Agg"; import torch, flashinfer, matplotlib; assert torch.cuda.is_available(); x=torch.ones((16,16),device="cuda",dtype=torch.bfloat16); y=x@x; torch.cuda.synchronize(); print(torch.__version__, torch.version.cuda, flashinfer.__version__, matplotlib.__version__, torch.cuda.get_device_name(0), float(y[0,0]))'
], text=True)
(RESULT_DIR / 'gpu_import_check.txt').write_text(import_check)
freeze = subprocess.check_output([UV, 'pip', 'freeze', '--python', str(VENV_PY)], text=True)
(RESULT_DIR / 'resolved_requirements.txt').write_text(freeze)
print('Environment and tests passed:', import_check.strip())

## 4. Reuse or download the pinned BAGEL checkpoint

The cache lives on Drive, so the first run downloads the checkpoint and later fresh runtimes reuse it. Server startup time is recorded but excluded from request timing.

In [ ]:
subprocess.run([
    str(VENV_PY), str(REPO / 'experiments/bagel/scripts/download_checkpoint.py'),
    '--cache-dir', str(CACHE_DIR),
    '--manifest', str(RESULT_DIR / 'checkpoint_manifest.json'),
], check=True)
import yaml
manifest = json.loads((RESULT_DIR / 'checkpoint_manifest.json').read_text())
pinned_config = yaml.safe_load((REPO / 'experiments/bagel/configs/bagel_single_gpu_pinned.yaml').read_text())
assert manifest['resolved_revision'] == pinned_config['model_kwargs']['checkpoint_revision']
print('Pinned checkpoint is available at', manifest['resolved_revision'])

## 5. Run the paired performance pilot

This starts one server, warms `reference` and `drop_image` once each, and runs five measured pairs. Even pairs run reference first; odd pairs run pruning first. Expect roughly **9–12 minutes after server startup** based on Step 4 timings. Do not rerun this cell while it is active.

In [ ]:
assert not BENCHMARK_DIR.exists(), f'Use a fresh RUN_ID: {BENCHMARK_DIR}'
with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as probe_socket:
    try:
        probe_socket.bind(('127.0.0.1', 8025))
    except OSError as exc:
        raise RuntimeError('Port 8025 is busy. Stop the earlier M* process first.') from exc
subprocess.run([
    str(VENV_PY), str(REPO / 'experiments/bagel/scripts/run_cfg_branch_benchmark.py'),
    '--repo-root', str(REPO),
    '--cache-dir', str(CACHE_DIR),
    '--output-dir', str(BENCHMARK_DIR),
    '--port', '8025',
    '--repeats', '5',
], cwd=REPO, check=True)
print('Paired CFG performance pilot completed:', BENCHMARK_DIR)

## 6. Inspect timing, exactness, and prepared branch rows

A successful analysis requires 49 traced refinement steps per request, the expected prepared branch labels at every step, and exact PNG hashes in all five pairs. Medians and interquartile ranges describe this pilot; five pairs are not a tail-latency study.

In [ ]:
orchestrator = json.loads((BENCHMARK_DIR / 'cfg_benchmark_orchestrator.json').read_text())
analysis = json.loads((BENCHMARK_DIR / 'cfg_branch_benchmark_analysis.json').read_text())
assert orchestrator['status'] == 'success', orchestrator
assert analysis['status'] == 'complete', analysis
rows = []
for policy in ('reference', 'drop_image'):
    item = analysis['policy_summary'][policy]
    rows.append({
        'policy': policy,
        'prepared_virtual_rows': item['observed_virtual_rows_per_request'],
        'median_client_seconds': round(item['client_seconds']['median'], 3),
        'client_iqr_seconds': [round(item['client_seconds']['q1'], 3), round(item['client_seconds']['q3'], 3)],
        'median_refinement_cuda_seconds': round(item['refinement_cuda_seconds']['median'], 3),
        'peak_sampled_gpu_memory_mib_median': item['peak_sampled_gpu_memory_mib']['median'],
    })
paired = analysis['paired_comparison']
print(json.dumps({
    'resolved_commit': head,
    'server_ready_seconds': orchestrator['server_ready_seconds'],
    'policies': rows,
    'exact_png_pairs': paired['exact_png_pair_count'],
    'median_client_speedup': paired['client_speedup']['median'],
    'client_speedup_iqr': [paired['client_speedup']['q1'], paired['client_speedup']['q3']],
    'median_refinement_cuda_speedup': paired['refinement_cuda_speedup']['median'],
    'measurement_boundary': analysis['measurement_boundary'],
    'limitations': analysis['limitations'],
    'result_dir': str(BENCHMARK_DIR),
}, indent=2))
from IPython.display import Image, display
display(Image(filename=str(BENCHMARK_DIR / 'cfg_branch_benchmark_analysis.png'), width=1000))

## 7. Package the evidence

Download the ZIP before deleting the runtime. It contains raw execution traces, per-request JSON and PNG files, GPU samples, logs, environment provenance, analysis JSON, and the plot.

In [ ]:
archive_base = RESULT_DIR.parent / f'{RUN_ID}_results'
archive_path = shutil.make_archive(
    str(archive_base), 'zip', root_dir=RESULT_DIR.parent, base_dir=RESULT_DIR.name
)
print('Saved ZIP:', archive_path)
from google.colab import files
files.download(archive_path)

After the ZIP download finishes, use **Runtime → Disconnect and delete runtime**. Share the printed summary and ZIP. The next decision will be based on paired eager timing stability; compiled execution and combination with heterogeneous continuous batching remain separate gates.